In [3]:
import sys
from pathlib import Path

# Find the project root directory by climbing up from the notebook location
project_root = Path().resolve()
while project_root.name and project_root.name != "baligh":
    if (project_root / "src").exists():
        break
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root added to sys.path: {project_root}")

Project root added to sys.path: /home/akramhany/Akram/University/GP/baligh


In [4]:
from src.core.schemas import Token
from src.services.nws.config import load_nws_config
from src.services.nws.features.cache.idioms import IdiomsCache
from src.services.nws.features.cache.manager import CacheManager
from src.services.nws.features.cache.phrases import PhrasesCache
from src.services.nws.features.cache.user_lru import UserLRUCache

# Load configuration
config = load_nws_config()

# Initialize caches
idioms_cache = IdiomsCache(config.cache.resolved_idioms_path)
phrases_cache = PhrasesCache(config.cache.resolved_phrases_path)
user_lru = UserLRUCache(maxsize=config.cache.user_lru_maxsize)

print(config.cache.resolved_idioms_path)
print(config.cache.resolved_phrases_path)

# Initialize CacheManager
cache_manager = CacheManager(
    tier1=idioms_cache,
    tier2=phrases_cache,
    tier3=user_lru,
    context_window_size=config.context_window_size,
)

print("Cache layer loaded successfully!")
print(f"- Tier 1 (Idioms): {len(idioms_cache._cache)} keys loaded.")
print(f"- Tier 2 (Phrases): {len(phrases_cache._cache)} keys loaded.")
print(f"- Tier 3 (User LRU): {len(user_lru)} items cached.")

/home/akramhany/Akram/University/GP/baligh/src/services/nws/data/idioms.yaml
/home/akramhany/Akram/University/GP/baligh/src/services/nws/data/phrases.yaml
Cache layer loaded successfully!
- Tier 1 (Idioms): 1988 keys loaded.
- Tier 2 (Phrases): 187 keys loaded.
- Tier 3 (User LRU): 0 items cached.


In [5]:
def make_tokens(text: str) -> list[Token]:
    """Converts a space-separated string into a list of Token objects."""
    words = text.split()
    return [
        Token(index=i, form=w, span=(0, len(w)), norm_span=(0, len(w)))
        for i, w in enumerate(words)
    ]


def test_nws_cache(context: str, fragment: str | None = None):
    """Helper to run cache lookup and display formatted results."""
    tokens = make_tokens(context)
    key = cache_manager.build_key(tokens, fragment)
    print(f"Input Context: '{context}'")
    if fragment is not None:
        print(f"Fragment:      '{fragment}'")
    print(f"Cache Key:      '{key}'")

    suggestions = cache_manager.lookup(key)
    if suggestions:
        source = suggestions[0].source
        print(f"\nFound {len(suggestions)} suggestion(s) in {source.upper()}")
        for s in suggestions:
            print(f"- Suggestion: {s.word:<15} | Score: {s.score:.2f} | Rank: {s.rank}")
    else:
        print("\nMISS: No cached suggestion found.")
    print("-" * 50)

In [13]:
custom_context = input(
    "Enter context (e.g. 'الحمد لله رب' or 'ابتسم عند الهزيمه'): "
).strip()
custom_fragment = input("Enter fragment (or press enter to skip): ").strip() or None

if custom_context:
    test_nws_cache(custom_context, custom_fragment)
else:
    print("Please provide a context to test!")

Input Context: 'شيبشيسب'
Fragment:      'بب'
Cache Key:      'شيبشيسب|بب'

MISS: No cached suggestion found.
--------------------------------------------------
